# 02 - Generowanie datasetu RAFT

Ten notebook generuje zbiór treningowy RAFT za pomocą Gemini Pro:
- Generowanie pytań (Q) z golden docs
- Synteza odpowiedzi Chain-of-Thought (A*)
- Kompozycja kontekstów z regułą P=80%

**Wymaga:** Gemini API key (darmowy tier: 15 RPM, 1M tokens/day)

In [ ]:
%%capture
!pip install google-generativeai tqdm

In [ ]:
import os
import sys
import json
sys.path.insert(0, "../src")

from dataset_builder import (
    get_gemini_model,
    chunk_document,
    build_raft_dataset,
    save_raft_dataset,
    load_raft_dataset,
    format_for_training,
)
from data_collection import load_distractors_from_jsonl

# ⚠️ Ustaw swój klucz API
os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"  # <- uzupełnij

## 1. Ładowanie danych

In [ ]:
# Załaduj dane z Fazy 1
golden_docs = load_distractors_from_jsonl("../data/golden_docs.jsonl")
distractors = load_distractors_from_jsonl("../data/distractors.jsonl")

print(f"Golden docs: {len(golden_docs)}")
print(f"Dystraktory: {len(distractors)}")

## 2. Chunking golden docs

In [ ]:
# Chunk golden docs (dystraktory są już krótkie)
chunked_golden = []
for doc in golden_docs:
    chunks = chunk_document(doc, chunk_size=400, chunk_overlap=50)
    chunked_golden.extend(chunks)

print(f"Golden docs po chunkingu: {len(chunked_golden)} fragmentów")
print(f"Średnia długość: {sum(len(d['content']) for d in chunked_golden) / len(chunked_golden):.0f} znaków")

## 3. Inicjalizacja Gemini

In [ ]:
model = get_gemini_model()

# Test: czy model odpowiada
test_response = model.generate_content("Odpowiedz jednym słowem: czy 2+2=4?")
print(f"Gemini test: {test_response.text}")

## 4. Generowanie datasetu RAFT

⚠️ Ten krok trwa 20-40 minut (rate limiting Gemini free tier: 15 RPM)

In [ ]:
raft_dataset = build_raft_dataset(
    golden_docs=chunked_golden,
    distractors_pool=distractors,
    model=model,
    oracle_ratio=0.8,           # P = 80%
    n_questions_per_doc=3,      # 3 pytania per chunk
    n_distractors=4,            # 4 dystraktory w kontekście
    delay=5.0,                  # 5s delay (respektuj rate limits)
    use_synthetic_distractors=True,
)

print(f"\nWygenerowano {len(raft_dataset)} przykładów RAFT")

## 5. Walidacja i podział train/test

In [ ]:
import random

# Statystyki
n_with_oracle = sum(1 for d in raft_dataset if d["has_oracle"])
n_without = len(raft_dataset) - n_with_oracle
print(f"Z wyrocznią: {n_with_oracle} ({n_with_oracle/len(raft_dataset):.0%})")
print(f"Bez wyroczni: {n_without} ({n_without/len(raft_dataset):.0%})")

# Podział train/test (80/20)
random.seed(42)
random.shuffle(raft_dataset)

split_idx = int(len(raft_dataset) * 0.8)
train_data = raft_dataset[:split_idx]
test_data = raft_dataset[split_idx:]

print(f"\nTrain: {len(train_data)} przykładów")
print(f"Test: {len(test_data)} przykładów")

In [ ]:
# Zapis
save_raft_dataset(train_data, "../data/raft_train.jsonl")
save_raft_dataset(test_data, "../data/raft_test.jsonl")

print("\nDane gotowe do fine-tuningu. Przejdź do notebook 03_finetuning.")

## 6. Podgląd wygenerowanych przykładów

In [ ]:
# Pokaż 2 przykłady
for i, example in enumerate(train_data[:2]):
    print(f"\n{'='*60}")
    print(f"  PRZYKŁAD {i+1} (oracle: {example['has_oracle']})")
    print(f"{'='*60}")
    print(f"\n📋 Pytanie: {example['question']}")
    print(f"\n📄 Kontekst (pierwsze 300 zn.): {example['context'][:300]}...")
    print(f"\n✅ Odpowiedź CoT (pierwsze 300 zn.): {example['answer'][:300]}...")

In [ ]:
# Pokaż sformatowany prompt treningowy
print("PROMPT TRENINGOWY:")
print("="*60)
formatted = format_for_training(train_data[0])
print(formatted[:800])
print("...")